In [10]:
from langchain.memory import ConversationSummaryBufferMemory
from langchain.chat_models import ChatOpenAI
from langchain.schema.runnable import RunnablePassthrough
from langchain.prompts import ChatPromptTemplate, MessagesPlaceholder

llm = ChatOpenAI(
    temperature=0.1,
    # model="gpt-4o-mini",
)

memory = ConversationSummaryBufferMemory(
    llm=llm,
    max_token_limit=70,
    return_messages=True,
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "you are a helpful AI talking to a human"),
        MessagesPlaceholder(variable_name="history"),
        ("human", "{question}"),
    ]
)


def load_memory(_):
    return memory.load_memory_variables({})["history"]


chain = RunnablePassthrough.assign(history=load_memory) | prompt | llm


def invoke_chain(question):
    result = chain.invoke({"question": question})
    memory.save_context(
        {"input": question},
        {"output": result.content},
    )
    print(result)

In [11]:
invoke_chain("my name is teo?")

content='Hello Teo! How can I assist you today?'


In [12]:
invoke_chain("what is my name?")

content='Your name is Teo. How can I assist you today, Teo?'
